In [4]:
import getpass
from dotenv import load_dotenv
import os

In [9]:
load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv('Gemini_Api_Key')

In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)
llm

ChatGoogleGenerativeAI(model='models/gemini-2.0-flash', google_api_key=SecretStr('**********'), temperature=0.0, max_retries=2, client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 0x0000013343187A90>, default_metadata=(), model_kwargs={})

In [12]:
#test message

In [13]:
messages = [
    (
        "system",
        "You are a helpful assistant that translates English to French. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]
ai_msg = llm.invoke(messages)
ai_msg

AIMessage(content="J'adore la programmation.", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--7b2487b9-a572-4aa0-acf1-e57e9a96f663-0', usage_metadata={'input_tokens': 20, 'output_tokens': 7, 'total_tokens': 27, 'input_token_details': {'cache_read': 0}})

In [14]:
print(ai_msg.content)

J'adore la programmation.


In [15]:
#lets work with the same prompt

In [16]:
patient_symptons = ['Fever' , 'Headache' , 'Soreness']

In [17]:
patient_condition = {
    "Age": 67,
    "Mobility Issues": "Requires cane for walking; limited stamina when standing for extended periods",
    "Known Allergies": ["Penicillin", "Latex"],
    "Chronic Conditions": ["Type 2 Diabetes", "Mild Hypertension"],
    "Recent Surgeries": "Hip replacement (8 months ago)",
    "Immunization Status": "Up-to-date, including flu and COVID-19 boosters",
    "Other Notes": "Reports occasional dizziness and sensitivity to cold weather"
}
patient_condition

{'Age': 67,
 'Mobility Issues': 'Requires cane for walking; limited stamina when standing for extended periods',
 'Known Allergies': ['Penicillin', 'Latex'],
 'Chronic Conditions': ['Type 2 Diabetes', 'Mild Hypertension'],
 'Recent Surgeries': 'Hip replacement (8 months ago)',
 'Immunization Status': 'Up-to-date, including flu and COVID-19 boosters',
 'Other Notes': 'Reports occasional dizziness and sensitivity to cold weather'}

In [18]:
geographic_location = {
    "Country": "United States",
    "State": "Kentucky",
    "City": "Highland Heights",
    "Zip Code": "41076",
    "Coordinates": {
        "Latitude": 39.1311,
        "Longitude": -84.5083
    },
    "Preferred Mode of Care": "In person"
}

In [19]:
from geopy.distance import geodesic
import pandas as pd

In [20]:
# Sample list of hospitals with coordinates (for demo purposes)
hospital_data = [
    {"Name": "University of Cincinnati Medical Center", "Latitude": 39.1412, "Longitude": -84.5059},
    {"Name": "Cincinnati VA Medical Center", "Latitude": 39.1398, "Longitude": -84.5039},
    {"Name": "Christ Hospital", "Latitude": 39.1231, "Longitude": -84.5070},
    {"Name": "Cleveland Clinic", "Latitude": 41.5033, "Longitude": -81.6200},
    {"Name": "Mount Carmel East", "Latitude": 39.9828, "Longitude": -82.8291}
]
#make a dataframe of this, in reality this will I think be one dataframe per state, consisteing of all health care and hospital services in that area
df_hospitals = pd.DataFrame(hospital_data)


In [21]:
print("df_hospitals can also include things like what services are offered, which department and nuimber of employees; this is bascially a very big table")

df_hospitals can also include things like what services are offered, which department and nuimber of employees; this is bascially a very big table


In [22]:
#One example of  us calculating distance from patient to each hospital
latitude = geographic_location["Coordinates"]["Latitude"]
longitude = geographic_location["Coordinates"]["Longitude"]
patient_coords = (latitude, longitude)
patient_coords

(39.1311, -84.5083)

In [23]:
distances = []
# Calculate distance for each hospital
for index, row in df_hospitals.iterrows():
    hospital_coords = (row["Latitude"], row["Longitude"])
    distance = geodesic(patient_coords, hospital_coords).miles
    distances.append(distance)
distances

[0.7085628278211792,
 0.6450317424197475,
 0.5562695371259393,
 223.71261059390406,
 107.20597664086802]

In [24]:
df_hospitals["Distance_miles"] = distances
df_hospitals

,Name,Latitude,Longitude,Distance_miles
0,University of Cincinnati Medical Center,39.1412,-84.5059,0.708563
1,Cincinnati VA Medical Center,39.1398,-84.5039,0.645032
2,Christ Hospital,39.1231,-84.5070,0.556270
3,Cleveland Clinic,41.5033,-81.6200,223.712611
4,Mount Carmel East,39.9828,-82.8291,107.205977


In [25]:
closest_index = df_hospitals["Distance_miles"].idxmin() #find the id of the minimum
closest_hospital = df_hospitals.loc[closest_index, "Name"] #look up the name of the minimum
print("The closest hospital is", closest_hospital)
#Actually, gemini is smart enough for now to choose a hoispital for us based on the values as well.

The closest hospital is Christ Hospital


In [26]:
prompt = f'''
You are a healthcare AI assistant tasked with generating a customized treatment plan for a patient. Given the following inputs:
1. List of patient symptoms which is {patient_symptons}
2. Patient's physical condition which is {patient_condition}
3. Geographic location which is {geographic_location}
Also the list of hospitals are listed below: {df_hospitals}
Generate a detailed treatment plan that includes the following:

1.Suggested medical actions: Specify appropriate steps such as scheduling a primary care visit, recommending lab tests or imaging, referrals to specialists, or urgent care if needed.
2.Location-specific considerations: Based on the patient’s location, recommend nearby clinics, hospitals, or telehealth options within a reasonable distance.
3.Justifications: For every suggested action, include a clear, medically-sound justification based on the patient’s symptoms and conditions.

Format your output in three clearly labeled sections:
1. Medical Actions
2. Location-Specific Options
3. Justifications

Be concise, medically accurate, and context-aware. Avoid unnecessary medical jargon.

'''
prompt

"\nYou are a healthcare AI assistant tasked with generating a customized treatment plan for a patient. Given the following inputs:\n1. List of patient symptoms which is ['Fever', 'Headache', 'Soreness']\n2. Patient's physical condition which is {'Age': 67, 'Mobility Issues': 'Requires cane for walking; limited stamina when standing for extended periods', 'Known Allergies': ['Penicillin', 'Latex'], 'Chronic Conditions': ['Type 2 Diabetes', 'Mild Hypertension'], 'Recent Surgeries': 'Hip replacement (8 months ago)', 'Immunization Status': 'Up-to-date, including flu and COVID-19 boosters', 'Other Notes': 'Reports occasional dizziness and sensitivity to cold weather'}\n3. Geographic location which is {'Country': 'United States', 'State': 'Kentucky', 'City': 'Highland Heights', 'Zip Code': '41076', 'Coordinates': {'Latitude': 39.1311, 'Longitude': -84.5083}, 'Preferred Mode of Care': 'In person'}\nAlso the list of hospitals are listed below:                                       Name  Latitu

In [32]:
messages = [
    (
        "system",
        "You are a healthcare AI assistant tasked with generating a customized treatment plan for a patient.",
    ),
    ("human", prompt),
]
ai_msg = llm.invoke(messages)
response = ai_msg.content
response

"Okay, here's a treatment plan based on the provided patient information:\n\n**1. Medical Actions**\n\n*   **Schedule a Primary Care Visit:** Within the next 24-48 hours.\n*   **Recommend Lab Tests:** Complete Blood Count (CBC), Comprehensive Metabolic Panel (CMP), and Influenza/COVID-19 PCR test.\n*   **Monitor Blood Pressure and Blood Sugar:** Patient should monitor these at home twice daily and record the results.\n*   **Symptomatic Relief:** Recommend over-the-counter acetaminophen or ibuprofen for fever and headache, being mindful of potential interactions with existing medications and conditions.\n\n**2. Location-Specific Options**\n\n*   **Primary Care Visit:** Given the patient's preference for in-person care and location in Highland Heights, KY, recommend scheduling an appointment with a primary care physician at Christ Hospital.\n*   **Urgent Care Alternative:** If a primary care appointment isn't immediately available, consider visiting the Cincinnati VA Medical Center.\n*  

In [33]:
# Display the output in a clean, formatted manner
print("\n===== Customized Treatment Plan =====\n")

# Split sections if the model followed the prompt structure
sections = response.strip().split('\n\n')

for section in sections:
    if section.strip():
        print(section)
        print("-" * 60)  # Divider between sections


===== Customized Treatment Plan =====

Okay, here's a treatment plan based on the provided patient information:
------------------------------------------------------------
**1. Medical Actions**
------------------------------------------------------------
*   **Schedule a Primary Care Visit:** Within the next 24-48 hours.
*   **Recommend Lab Tests:** Complete Blood Count (CBC), Comprehensive Metabolic Panel (CMP), and Influenza/COVID-19 PCR test.
*   **Monitor Blood Pressure and Blood Sugar:** Patient should monitor these at home twice daily and record the results.
*   **Symptomatic Relief:** Recommend over-the-counter acetaminophen or ibuprofen for fever and headache, being mindful of potential interactions with existing medications and conditions.
------------------------------------------------------------
**2. Location-Specific Options**
------------------------------------------------------------
*   **Primary Care Visit:** Given the patient's preference for in-person care and l